# Analyse PE samples

In [ ]:
# define PE run settings
test_case  = ''    # which test case to load: '', '_amps_off', '_evolution_primary_off'
inj = '1PA'
rec = '1PA'
eps = 'e-5'
discard= 0   # burn-in iterations to discard

# Post-processing
do_filter_lost   = False  # remove lost walkers
do_thin          = False  # thin by integrated auto-correlation time to exclude correlated samples
exclude_sky      = False  # exclude sky-angle params from the intrinsic corner plot

# Threshold for lost walker detection 
# This only applies to the cold chain
lost_lower_bound = -30.0

# Colour for this run's posterior in overlay plots
posterior_color  = 'steelblue'

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.size'] = 14

from src.io import param_load
from src.analysis import (
    sampled_params_from_config,
    load_samples,
    plot_log_like,
    cut_samples_autocorr,
    filter_lost_walkers,
    corner_plot,
    plot_sky_position,
    plot_chain_convergence,
    plot_covariance_evolution,
    plot_gelman_rubin,
    plot_seaborn_diagnostics,
)

In [ ]:
if test_case is not None:
    config = param_load(f'./config/config_test{test_case}.yaml')
else:
    config = param_load(f'./config/config_inj_{inj}_rec_{rec}_eps_{eps}.yaml')

emri   = config['Injection']['EMRI']
inj_wf = config['Injection']['Waveform']
rec_wf = config['Recovery']['Waveform']

print(f"Injection model : {inj_wf['model']}")
print(f"Recovery model  : {rec_wf['model']}")
print(f"Fixed params    : {config['Sampler'].get('fixed_params', [])}")
print()
print('EMRI injection parameters')
print('=' * 32)
for k, v in emri.items():
    print(f'  {k:<16} {v}')

In [ ]:
param_names, true_vals = sampled_params_from_config(config)

print(f'Sampled parameters ({len(param_names)}):')
for name, val in zip(param_names, true_vals):
    print(f'  {name:<16} {val}')

In [ ]:
sampling_dir = f'{os.getcwd()}/data/sampling_data'
# select sampling files matching test case description
pattern      = os.path.join(sampling_dir, f'SamplingResults_test_{test_case}_*.h5')
# sort files by timestamp in filename
files        = sorted(glob.glob(pattern))

if not files:
    raise FileNotFoundError(f'No sampling file found matching {pattern}')
# Get last file, assuming this is the one we intend to analyze. 
file_path = files[-1]
print(f'Loading: {file_path}')

# Load samples and log-likelihoods from the HDF5 backend file
samples_per_temp, reader, log_like, (N_iters, N_temps, N_walkers, N_params) = \
    load_samples(file_path, discard=discard)

print(f'Shape: {N_iters} iters  x  {N_temps} temps  x  {N_walkers} walkers  x  {N_params} params')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_log_like(log_like, temps=list(range(N_temps)), discard=discard, ax=ax)
plt.show()

In [ ]:
# if the lgglikelihood shows lost walkers, we filetr these out from the cold chain
if do_filter_lost:
    clean_samples, kept_idx, ll_clean = filter_lost_walkers(
        reader, discard=discard, lower_bound=lost_lower_bound
    )
    # Replace cold-chain samples with the filtered version
    samples_per_temp[0] = clean_samples
    print(f'Cold chain after filtering: {clean_samples.shape}')
else:
    print('Skipping lost-walker filtering.')

In [ ]:
if do_thin:
    samples_per_temp = cut_samples_autocorr(reader, samples_per_temp, discard=discard)
    print(f'Thinned cold chain: {samples_per_temp[0].shape}')
else:
    print('Skipping autocorrelation thinning.')

## Corner plot
Create corner plot kto inspect the posterior distributions of parameters and their correlations

In [ ]:
# samples_dict: key = legend label; value = dict with 'color' and 'samples'.
# Add extra entries here to overlay additional posteriors (e.g. higher temps).
samples_dict = {
    'Test run': {
        'color':   posterior_color,
        'samples': samples_per_temp[0],
    },
}

os.makedirs(f'{os.getcwd()}/Plots', exist_ok=True)
plot_name_intrinsic = f'{os.getcwd()}/Plots/test_{test_case}_corner_intrinsic'

fig = corner_plot(
    samples_dict,
    true_vals,
    param_names,
    inj_model=inj_wf['model'],
    rec_model=rec_wf['model'],
    exclude_sky=True,          # intrinsic + phases only
    plot_name=plot_name_intrinsic,
)
plt.show()

In [ ]:
# Full corner plot including sky-angle parameters
plot_name_full = f'{os.getcwd()}/Plots/test_{test_case}_corner_full'

fig = corner_plot(
    samples_dict,
    true_vals,
    param_names,
    inj_model=inj_wf['model'],
    rec_model=rec_wf['model'],
    exclude_sky=False,
    plot_name=plot_name_full,
)
plt.show()

## Sky position

In [ ]:
theta_S_true = float(emri['theta_S']) if 'theta_S' in emri else None
phi_S_true   = float(emri['phi_S'])   if 'phi_S'   in emri else None

plot_name_sky = f'{os.getcwd()}/Plots/test_{test_case}_sky_position'

fig = plot_sky_position(
    samples_dict,
    param_names,
    true_theta_S=theta_S_true,
    true_phi_S=phi_S_true,
    plot_name=plot_name_sky,
)

## Verify chain convergence
A stable loglikelihood trace indicates that the chains have converged. However, proper convergence diagnostics are needed to confirm this. We plot the loglikelihood trace for the cold chain (T=0) to visually inspect its behavior over iterations. A stable and well-mixed trace suggests good convergence, while trends or large fluctuations point to issues with convergence.

In [ ]:
plot_name_traces = f'{os.getcwd()}/Plots/test_{test_case}_traces'

fig = plot_chain_convergence(
    reader, param_names,
    discard=discard, temp=0,
    plot_name=plot_name_traces,
)

In [ ]:
plot_name_cov = f'{os.getcwd()}/Plots/test_{test_case}_covariance'

fig = plot_covariance_evolution(
    reader, param_names,
    discard=discard, temp=0, step=50,
    plot_name=plot_name_cov,
)

In [ ]:
plot_name_rhat = f'{os.getcwd()}/Plots/test_{test_case}_gelman_rubin'

rhat = plot_gelman_rubin(
    reader, param_names,
    discard=discard, temp=0, step=50,
    convergence_threshold=1.01,
    plot_name=plot_name_rhat,
)
print('Final R-hat values:')
for name, rh in zip(param_names, rhat[-1]):
    print(f'  {name:<16} {rh:.4f}')

## Statistical diagnostics
Use the statistical tools in seaborn to inspect properties of the inferred posterior distributions, such as skewness, multimodality, etc. This can help identify potential issues with the sampling process or the model fit.

In [ ]:
plot_name_sb = f'{os.getcwd()}/Plots/test_{test_case}_seaborn'

title = (
    f'test_{test_case}  |  '
    f"Inj: {inj_wf['model']}  Rec: {rec_wf['model']}"
)

fig = plot_seaborn_diagnostics(
    samples_per_temp[0],
    param_names,
    title=title,
    max_pair_params=6,
    plot_name=plot_name_sb,
)